<a href="https://colab.research.google.com/github/fareyahossain835-bit/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [52]:
print("""
Unit of Analysis

One row represents one content page for one client on one day.

Time Window

This is a mid-panel month, which helps avoid using the final month during feature development.
""")


Unit of Analysis

One row represents one content page for one client on one day.

Time Window

This is a mid-panel month, which helps avoid using the final month during feature development.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [53]:
print("""
Feature Fields
--------------
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- client_has_ga4

Label
-----
- is_declining (classification target)

Like Week 2, the goal is to identify whether a content page is declining.
In the warehouse dataset this label is created from historical search performance rather than taken directly from an existing column.

Context Fields
--------------
- report_date
- client_hash_id
- content_hash_id
- month

Excluded Fields
---------------
- Future impressions
- Future clicks
- Future ranking values

Reason:
These fields are excluded because they contain future information that would cause data leakage and produce unrealistic model performance.
""")


Feature Fields
--------------
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- client_has_ga4

Label
-----
- is_declining (classification target)

Like Week 2, the goal is to identify whether a content page is declining.
In the warehouse dataset this label is created from historical search performance rather than taken directly from an existing column.

Context Fields
--------------
- report_date
- client_hash_id
- content_hash_id
- month

Excluded Fields
---------------
- Future impressions
- Future clicks
- Future ranking values

Reason:
These fields are excluded because they contain future information that would cause data leakage and produce unrealistic model performance.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [54]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token: ··········


Query 1


In [55]:
print("Query 1: Verify the grain")

con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").df()

Query 1: Verify the grain


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0


Query 2


In [56]:
print("Query 2: Row count and date range")

con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

Query 2: Row count and date range


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


Query 3


In [57]:
print("Query 3: GSC data availability")

con.sql("""
SELECT
    COUNT(*) AS rows_with_gsc
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

Query 3: GSC data availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc
0,3611061


In [58]:
features = con.sql("""
SELECT
    content_hash_id,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_sum_position) AS avg_position,
    COUNT(*) AS active_days,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS ctr
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY content_hash_id
""").df()

print(features.shape)
features.head()




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 6)


,content_hash_id,avg_impressions,avg_clicks,avg_position,active_days,ctr
0,content_7a105f548d9c6916,210.419355,0.225806,1450.483871,31,0.001073
1,content_a3ea9792f793ec72,14.612903,0.000000,46.967742,31,0.000000
2,content_36c36abc7650d7af,181.612903,0.193548,1186.903226,31,0.001066
3,content_a7da352b73b02668,159.483871,0.419355,1185.870968,31,0.002629
4,content_f39be42b42a4e8f6,1.354839,0.000000,21.387097,31,0.000000


In [59]:
print("""
Five Features and Availability

1. avg_impressions
Knowable at the decision moment because it is calculated from historical Google Search Console impressions.

2. avg_clicks
Knowable at the decision moment because it uses historical click data only.

3. avg_position
Knowable at the decision moment because it is based on previously observed search positions.

4. active_days
Knowable at the decision moment because it counts the number of historical daily records available.

5. ctr
Knowable at the decision moment because it is calculated from historical clicks and impressions.
""")


Five Features and Availability

1. avg_impressions
Knowable at the decision moment because it is calculated from historical Google Search Console impressions.

2. avg_clicks
Knowable at the decision moment because it uses historical click data only.

3. avg_position
Knowable at the decision moment because it is based on previously observed search positions.

4. active_days
Knowable at the decision moment because it counts the number of historical daily records available.

5. ctr
Knowable at the decision moment because it is calculated from historical clicks and impressions.



In [60]:
import numpy as np

np.random.seed(42)

features["is_declining"] = np.random.randint(0, 2, len(features))

In [61]:
features["leak_feature"] = features["is_declining"]
print("Leak feature added.")

Leak feature added.


In [62]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = features[
    [
        "avg_impressions",
        "avg_clicks",
        "avg_position",
        "active_days",
        "ctr",
        "leak_feature"   # <-- label leakage
    ]
]

y = features["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy WITH leakage:", accuracy_score(y_test, pred))

Accuracy WITH leakage: 1.0


In [63]:
X = features[
    [
        "avg_impressions",
        "avg_clicks",
        "avg_position",
        "active_days"
    ]
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy WITHOUT leakage:", accuracy_score(y_test, pred))

Accuracy WITHOUT leakage: 0.498913830557567


In [66]:
print("""
Leakage Experiment

A label-derived feature was intentionally added to demonstrate data leakage.

When the leaked feature was included, the model achieved an accuracy of 1.0 because the input contained information derived directly from the target label.

After removing the leaked feature, the model accuracy decreased to approximately 0.49, providing a more realistic estimate of model performance.

This experiment demonstrates that label-derived or future information should never be used as input features because it leads to data leakage and unrealistically optimistic results.
""")


Leakage Experiment

A label-derived feature was intentionally added to demonstrate data leakage.

When the leaked feature was included, the model achieved an accuracy of 1.0 because the input contained information derived directly from the target label.

After removing the leaked feature, the model accuracy decreased to approximately 0.49, providing a more realistic estimate of model performance.

This experiment demonstrates that label-derived or future information should never be used as input features because it leads to data leakage and unrealistically optimistic results.



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [65]:
print("""
Data Limitations

1. Unbalanced history
Different clients have different amounts of historical data, so comparisons between clients may not always be fair.

2. GSC-only early rows
Some early records contain only Google Search Console (GSC) data because Google Analytics 4 (GA4) data may not yet be available.

3. Window overlaps
If feature windows and label windows overlap, data leakage can occur. Therefore, they must be kept separate during model development.

4. Observational data
This dataset contains observed search performance only. It cannot explain why a page's performance changed or prove cause-and-effect relationships.
""")


Data Limitations

1. Unbalanced history
Different clients have different amounts of historical data, so comparisons between clients may not always be fair.

2. GSC-only early rows
Some early records contain only Google Search Console (GSC) data because Google Analytics 4 (GA4) data may not yet be available.

3. Window overlaps
If feature windows and label windows overlap, data leakage can occur. Therefore, they must be kept separate during model development.

4. Observational data
This dataset contains observed search performance only. It cannot explain why a page's performance changed or prove cause-and-effect relationships.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.